# Portfolio Diversification Engine Demonstration

This notebook demonstrates the Portfolio Diversification Engine with configurable YAML parameters.

## Features Demonstrated:
- YAML configuration loading
- HHI (Herfindahl-Hirschman Index) calculation
- Portfolio diversification scoring
- Concentration limit violations
- Risk adjustments for lending
- MCP service integration
- Historical analysis storage

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add the parent directory to Python path
sys.path.append(str(Path.cwd().parent))

# Import our modules
from core.config import PortfolioDiversificationConfig, load_config
from core.portfolio_engine import (
    PortfolioDiversificationEngine,
    AssetPosition,
    Portfolio,
    create_sample_portfolio
)

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Imports completed successfully!")

## 1. Configuration Loading

First, let's load and examine the YAML configuration:

In [ ]:
# Load configuration
config = load_config()
engine = PortfolioDiversificationEngine(config)

print("🔧 Configuration Summary:")
print(f"├── MCP Algorand Reader: {config.mcp_services.algorand_reader_url}")
print(f"├── MCP Market Data:     {config.mcp_services.market_data_url}")
print(f"├── Min HHI:             {config.hhi_calculation.min_hhi}")
print(f"├── Max HHI:             {config.hhi_calculation.max_hhi}")
print(f"├── Single Asset Max:    {config.concentration_limits.single_asset_max:.1%}")
print(f"├── Single Type Max:     {config.concentration_limits.single_type_max:.1%}")
print(f"├── Low Liquidity Max:   {config.concentration_limits.low_liquidity_max:.1%}")
print(f"└── Database Path:       {config.database.default_path}")

print("\n📊 Diversification Thresholds:")
print(f"├── Excellent: {config.diversification_scoring.excellent_threshold:.1%}+")
print(f"├── Good:      {config.diversification_scoring.good_threshold:.1%}+")
print(f"├── Adequate:  {config.diversification_scoring.adequate_threshold:.1%}+")
print(f"└── Poor:      {config.diversification_scoring.poor_threshold:.1%}+")

## 2. Portfolio Creation and Analysis

Let's create several portfolios with different diversification levels:

In [ ]:
# Create portfolios with different diversification levels
portfolios = {}

# 1. Well-diversified portfolio
portfolios['Well Diversified'] = Portfolio(
    positions=[
        AssetPosition(symbol="BTC", asset_type="cryptocurrency", value_usd=20000, weight=0.20, volatility_30d=0.15, daily_volume_usd=25_000_000_000, liquidity_tier="high"),
        AssetPosition(symbol="USDC", asset_type="stablecoin", value_usd=20000, weight=0.20, volatility_30d=0.02, daily_volume_usd=5_000_000_000, liquidity_tier="high"),
        AssetPosition(symbol="UNI", asset_type="defi_token", value_usd=20000, weight=0.20, volatility_30d=0.25, daily_volume_usd=200_000_000, liquidity_tier="medium"),
        AssetPosition(symbol="APE", asset_type="nft", value_usd=20000, weight=0.20, volatility_30d=0.35, daily_volume_usd=50_000_000, liquidity_tier="low"),
        AssetPosition(symbol="COMP", asset_type="governance_token", value_usd=20000, weight=0.20, volatility_30d=0.30, daily_volume_usd=75_000_000, liquidity_tier="medium")
    ],
    total_value_usd=100000,
    timestamp=datetime.now()
)

# 2. Cryptocurrency-concentrated portfolio
portfolios['Crypto Concentrated'] = Portfolio(
    positions=[
        AssetPosition(symbol="BTC", asset_type="cryptocurrency", value_usd=60000, weight=0.60, volatility_30d=0.15, daily_volume_usd=25_000_000_000, liquidity_tier="high"),
        AssetPosition(symbol="ETH", asset_type="cryptocurrency", value_usd=30000, weight=0.30, volatility_30d=0.18, daily_volume_usd=15_000_000_000, liquidity_tier="high"),
        AssetPosition(symbol="USDC", asset_type="stablecoin", value_usd=10000, weight=0.10, volatility_30d=0.02, daily_volume_usd=5_000_000_000, liquidity_tier="high")
    ],
    total_value_usd=100000,
    timestamp=datetime.now()
)

# 3. Single asset portfolio (worst case)
portfolios['Single Asset'] = Portfolio(
    positions=[
        AssetPosition(symbol="BTC", asset_type="cryptocurrency", value_usd=100000, weight=1.00, volatility_30d=0.15, daily_volume_usd=25_000_000_000, liquidity_tier="high")
    ],
    total_value_usd=100000,
    timestamp=datetime.now()
)

# 4. Algorand ecosystem portfolio
portfolios['Algorand Ecosystem'] = Portfolio(
    positions=[
        AssetPosition(symbol="ALGO", asset_type="cryptocurrency", value_usd=40000, weight=0.40, volatility_30d=0.25, daily_volume_usd=150_000_000, liquidity_tier="high"),
        AssetPosition(symbol="USDC", asset_type="stablecoin", value_usd=25000, weight=0.25, volatility_30d=0.02, daily_volume_usd=5_000_000_000, liquidity_tier="high"),
        AssetPosition(symbol="PLANETS", asset_type="defi_token", value_usd=15000, weight=0.15, volatility_30d=0.40, daily_volume_usd=5_000_000, liquidity_tier="medium"),
        AssetPosition(symbol="CHOICE", asset_type="governance_token", value_usd=12000, weight=0.12, volatility_30d=0.35, daily_volume_usd=2_000_000, liquidity_tier="medium"),
        AssetPosition(symbol="ALGONODE_NFT", asset_type="nft", value_usd=8000, weight=0.08, volatility_30d=0.50, daily_volume_usd=100_000, liquidity_tier="low")
    ],
    total_value_usd=100000,
    timestamp=datetime.now()
)

print(f"📊 Created {len(portfolios)} test portfolios for analysis")

## 3. Diversification Analysis

Now let's analyze each portfolio and compare the results:

In [ ]:
# Analyze all portfolios
analyses = {}
results_data = []

for name, portfolio in portfolios.items():
    print(f"\n🔍 Analyzing: {name}")
    analysis = engine.analyze_portfolio(portfolio)
    analyses[name] = analysis
    
    # Collect data for comparison
    results_data.append({
        'Portfolio': name,
        'HHI Score': analysis.hhi_score,
        'Diversification Score': analysis.diversification_score,
        'Diversification Level': analysis.diversification_level,
        'Confidence Score': analysis.confidence_score,
        'Num Positions': len(analysis.portfolio.positions),
        'Violations': len(analysis.concentration_violations),
        'Recommendations': len(analysis.recommendations)
    })
    
    print(f"  HHI Score:           {analysis.hhi_score:.3f}")
    print(f"  Diversification:     {analysis.diversification_score:.1%} ({analysis.diversification_level})")
    print(f"  Confidence:          {analysis.confidence_score:.1%}")
    print(f"  Violations:          {len(analysis.concentration_violations)}")
    print(f"  Recommendations:     {len(analysis.recommendations)}")

# Create comparison DataFrame
df_results = pd.DataFrame(results_data)
print("\n📈 Analysis completed for all portfolios!")

## 4. Results Visualization

Let's visualize the analysis results:

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Portfolio Diversification Analysis Results', fontsize=16, fontweight='bold')

# 1. Diversification Score Comparison
axes[0, 0].bar(df_results['Portfolio'], df_results['Diversification Score'], 
               color=['green' if x >= 0.8 else 'orange' if x >= 0.6 else 'red' for x in df_results['Diversification Score']])
axes[0, 0].set_title('Diversification Scores', fontweight='bold')
axes[0, 0].set_ylabel('Score')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].axhline(y=0.8, color='green', linestyle='--', alpha=0.7, label='Excellent (80%)')
axes[0, 0].axhline(y=0.6, color='orange', linestyle='--', alpha=0.7, label='Good (60%)')
axes[0, 0].axhline(y=0.4, color='red', linestyle='--', alpha=0.7, label='Adequate (40%)')
axes[0, 0].legend()

# 2. HHI vs Diversification Score
scatter = axes[0, 1].scatter(df_results['HHI Score'], df_results['Diversification Score'], 
                            s=100, alpha=0.7, c=range(len(df_results)))
axes[0, 1].set_title('HHI vs Diversification Score', fontweight='bold')
axes[0, 1].set_xlabel('HHI Score (lower = more diversified)')
axes[0, 1].set_ylabel('Diversification Score')
for i, row in df_results.iterrows():
    axes[0, 1].annotate(row['Portfolio'], (row['HHI Score'], row['Diversification Score']), 
                       xytext=(5, 5), textcoords='offset points', fontsize=8)

# 3. Confidence Scores
axes[1, 0].bar(df_results['Portfolio'], df_results['Confidence Score'], 
               color=['darkgreen' if x >= 0.8 else 'gold' if x >= 0.6 else 'lightcoral' for x in df_results['Confidence Score']])
axes[1, 0].set_title('Confidence Scores', fontweight='bold')
axes[1, 0].set_ylabel('Confidence')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Issues Summary
width = 0.35
x = range(len(df_results))
axes[1, 1].bar([i - width/2 for i in x], df_results['Violations'], width, label='Violations', color='red', alpha=0.7)
axes[1, 1].bar([i + width/2 for i in x], df_results['Recommendations'], width, label='Recommendations', color='blue', alpha=0.7)
axes[1, 1].set_title('Issues and Recommendations', fontweight='bold')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(df_results['Portfolio'], rotation=45)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Display results table
print("\n📊 Detailed Results Comparison:")
display(df_results.round(3))

## 5. Detailed Analysis of Each Portfolio

Let's examine each portfolio in detail:

In [ ]:
def display_portfolio_details(name, analysis):
    """Display detailed analysis for a portfolio"""
    print(f"\n{'='*60}")
    print(f"📊 {name} - Detailed Analysis")
    print(f"{'='*60}")
    
    # Portfolio composition
    print(f"\n💼 Portfolio Composition:")
    for pos in analysis.portfolio.positions:
        print(f"  {pos.symbol:12} | {pos.asset_type:15} | ${pos.value_usd:8,.0f} | {pos.weight:6.1%} | {pos.liquidity_tier}")
    
    # Metrics
    print(f"\n📈 Diversification Metrics:")
    print(f"  Total Value:         ${analysis.portfolio.total_value_usd:,.0f}")
    print(f"  HHI Score:           {analysis.hhi_score:.3f}")
    print(f"  Diversification:     {analysis.diversification_score:.1%} ({analysis.diversification_level})")
    print(f"  Confidence Score:    {analysis.confidence_score:.1%}")
    
    # Issues
    if analysis.concentration_violations:
        print(f"\n⚠️  Concentration Violations:")
        for violation in analysis.concentration_violations:
            print(f"    • {violation}")
    
    if analysis.correlation_risks:
        print(f"\n⚠️  Correlation Risks:")
        for risk in analysis.correlation_risks:
            print(f"    • {risk}")
    
    if analysis.liquidity_risks:
        print(f"\n⚠️  Liquidity Risks:")
        for risk in analysis.liquidity_risks:
            print(f"    • {risk}")
    
    # Recommendations
    if analysis.recommendations:
        print(f"\n💡 Recommendations:")
        for rec in analysis.recommendations:
            print(f"    • {rec}")
    
    # Risk adjustments
    if analysis.risk_adjustments:
        print(f"\n⚖️  Risk Adjustments for Lending:")
        for key, value in analysis.risk_adjustments.items():
            icon = "📉" if "reduction" in key or "bonus" in key else "📈"
            print(f"    {icon} {key.replace('_', ' ').title()}: {value:+.1%}")

# Display details for each portfolio
for name, analysis in analyses.items():
    display_portfolio_details(name, analysis)

## 6. Asset Type Distribution Analysis

In [ ]:
# Create asset type distribution charts
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Asset Type Distribution by Portfolio', fontsize=16, fontweight='bold')

portfolio_names = list(portfolios.keys())
colors = plt.cm.Set3(range(12))  # More colors for asset types

for i, (name, portfolio) in enumerate(portfolios.items()):
    # Calculate asset type distribution
    type_dist = {}
    for pos in portfolio.positions:
        if pos.asset_type not in type_dist:
            type_dist[pos.asset_type] = 0
        type_dist[pos.asset_type] += pos.value_usd
    
    # Convert to percentages
    total_value = sum(type_dist.values())
    type_percentages = {k: v/total_value*100 for k, v in type_dist.items()}
    
    # Create pie chart
    row, col = i // 2, i % 2
    wedges, texts, autotexts = axes[row, col].pie(
        type_percentages.values(), 
        labels=type_percentages.keys(),
        autopct='%1.1f%%',
        colors=colors[:len(type_percentages)],
        startangle=90
    )
    
    axes[row, col].set_title(f'{name}\n(HHI: {analyses[name].hhi_score:.3f})', fontweight='bold')
    
    # Adjust text size
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

## 7. MCP Integration Test

Let's test the MCP service integration:

In [ ]:
# Test MCP service connectivity
import aiohttp

async def test_mcp_services():
    """Test MCP service connectivity"""
    print("🔗 Testing MCP Service Connectivity...")
    
    services = {
        "Algorand Reader": config.mcp_services.algorand_reader_url,
        "Market Data": config.mcp_services.market_data_url
    }
    
    results = {}
    
    for service_name, url in services.items():
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(f"{url}/health", timeout=aiohttp.ClientTimeout(total=5)) as response:
                    if response.status == 200:
                        results[service_name] = "✅ Available"
                    else:
                        results[service_name] = f"⚠️ Status {response.status}"
        except Exception as e:
            results[service_name] = f"❌ Unavailable ({str(e)[:50]}...)"
    
    return results

# Run MCP test
try:
    # Create new event loop for Jupyter
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    mcp_results = loop.run_until_complete(test_mcp_services())
    
    print("\n🌐 MCP Service Status:")
    for service, status in mcp_results.items():
        print(f"  {service:15}: {status}")
    
    if any("Available" in status for status in mcp_results.values()):
        print("\n💡 Some MCP services are available for real-time data enrichment!")
    else:
        print("\n💡 MCP services are offline. To test with live data:")
        print("   1. Start Algorand Reader MCP on port 8002")
        print("   2. Start Market Data MCP on port 8003")
        print("   3. Re-run this cell")
        
except Exception as e:
    print(f"⚠️ Could not test MCP services: {e}")
finally:
    loop.close()

## 8. Historical Analysis Storage

Let's examine the historical analysis data stored in the database:

In [ ]:
# Get historical analysis data
history = engine.get_analysis_history(limit=20)

if history:
    print(f"📚 Found {len(history)} historical analysis records")
    
    # Convert to DataFrame for analysis
    df_history = pd.DataFrame(history)
    
    # Display recent records
    print("\n📊 Recent Analysis History:")
    columns_to_show = ['timestamp', 'portfolio_value_usd', 'num_positions', 
                      'hhi_score', 'diversification_score', 'diversification_level', 'confidence_score']
    
    display_df = df_history[columns_to_show].copy()
    display_df['portfolio_value_usd'] = display_df['portfolio_value_usd'].apply(lambda x: f"${x:,.0f}")
    display_df['hhi_score'] = display_df['hhi_score'].round(3)
    display_df['diversification_score'] = display_df['diversification_score'].round(3)
    display_df['confidence_score'] = display_df['confidence_score'].round(3)
    
    display(display_df.head(10))
    
    # Plot historical trends if we have enough data
    if len(df_history) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Diversification score trend
        axes[0].plot(range(len(df_history)), df_history['diversification_score'], marker='o')
        axes[0].set_title('Diversification Score Trend')
        axes[0].set_xlabel('Analysis #')
        axes[0].set_ylabel('Diversification Score')
        axes[0].grid(True, alpha=0.3)
        
        # Portfolio value vs diversification
        scatter = axes[1].scatter(df_history['portfolio_value_usd'], df_history['diversification_score'], 
                                 c=df_history['confidence_score'], cmap='viridis', alpha=0.7)
        axes[1].set_title('Portfolio Value vs Diversification')
        axes[1].set_xlabel('Portfolio Value (USD)')
        axes[1].set_ylabel('Diversification Score')
        plt.colorbar(scatter, ax=axes[1], label='Confidence Score')
        
        plt.tight_layout()
        plt.show()
        
else:
    print("📝 No historical analysis data found. Analysis results are stored automatically.")
    print(f"   Database location: {config.database.default_path}")

## 9. Configuration Impact Analysis

Let's see how different configuration parameters affect the analysis:

In [ ]:
# Test different configuration scenarios
print("🔧 Configuration Impact Analysis")
print("="*50)

# Use the Algorand ecosystem portfolio for testing
test_portfolio = portfolios['Algorand Ecosystem']

# Test 1: Different HHI thresholds
print("\n1. Impact of HHI Threshold Changes:")

hhi_configs = [
    {"min_hhi": 0.1, "max_hhi": 1.0, "name": "Lenient (0.1-1.0)"},
    {"min_hhi": 0.2, "max_hhi": 1.0, "name": "Standard (0.2-1.0)"},
    {"min_hhi": 0.3, "max_hhi": 1.0, "name": "Strict (0.3-1.0)"}
]

for hhi_config in hhi_configs:
    # Create modified config
    test_config = PortfolioDiversificationConfig()
    test_config.hhi_calculation.min_hhi = hhi_config["min_hhi"]
    test_config.hhi_calculation.max_hhi = hhi_config["max_hhi"]
    
    # Create engine with modified config
    test_engine = PortfolioDiversificationEngine(test_config)
    
    # Analyze portfolio
    analysis = test_engine.analyze_portfolio(test_portfolio)
    
    print(f"  {hhi_config['name']:20} → Diversification: {analysis.diversification_score:.1%} ({analysis.diversification_level})")

# Test 2: Different concentration limits
print("\n2. Impact of Concentration Limit Changes:")

concentration_configs = [
    {"single_asset_max": 0.30, "name": "Strict (30% max)"},
    {"single_asset_max": 0.40, "name": "Standard (40% max)"},
    {"single_asset_max": 0.50, "name": "Lenient (50% max)"}
]

for conc_config in concentration_configs:
    # Create modified config
    test_config = PortfolioDiversificationConfig()
    test_config.concentration_limits.single_asset_max = conc_config["single_asset_max"]
    
    # Create engine with modified config
    test_engine = PortfolioDiversificationEngine(test_config)
    
    # Analyze portfolio
    analysis = test_engine.analyze_portfolio(test_portfolio)
    
    violations = len(analysis.concentration_violations)
    print(f"  {conc_config['name']:20} → Violations: {violations}, Confidence: {analysis.confidence_score:.1%}")

print("\n💡 Configuration parameters significantly impact analysis results!")
print("   Adjust YAML config file to customize for your use case.")

## 10. Summary and Conclusions

Let's summarize what we've learned:

In [ ]:
print("\n" + "="*70)
print("📊 PORTFOLIO DIVERSIFICATION ENGINE SUMMARY")
print("="*70)

print("\n🎯 Key Features Demonstrated:")
print("   ✅ YAML-based configuration system")
print("   ✅ HHI (Herfindahl-Hirschman Index) calculation")
print("   ✅ Multi-level diversification scoring")
print("   ✅ Concentration limit violation detection")
print("   ✅ Risk-adjusted lending parameters")
print("   ✅ MCP service integration capability")
print("   ✅ Historical analysis storage")
print("   ✅ Comprehensive recommendation system")

print("\n📈 Analysis Results Summary:")
for name, analysis in analyses.items():
    risk_adj = ""
    if analysis.risk_adjustments:
        if "collateral_ratio_reduction" in analysis.risk_adjustments:
            risk_adj = f" (↓{analysis.risk_adjustments['collateral_ratio_reduction']:.1%} collateral)"
        elif "collateral_ratio_increase" in analysis.risk_adjustments:
            risk_adj = f" (↑{analysis.risk_adjustments['collateral_ratio_increase']:.1%} collateral)"
    
    print(f"   {name:20} → {analysis.diversification_score:.1%} ({analysis.diversification_level}){risk_adj}")

print("\n🔧 Configuration Highlights:")
print(f"   • HHI Range:          {config.hhi_calculation.min_hhi} - {config.hhi_calculation.max_hhi}")
print(f"   • Single Asset Max:   {config.concentration_limits.single_asset_max:.1%}")
print(f"   • MCP Integration:    {config.mcp_services.algorand_reader_url}")
print(f"   • Database Storage:   {config.database.backup_enabled}")

print("\n💡 Key Insights:")
print("   1. Well-diversified portfolios get better lending terms")
print("   2. Single-asset portfolios face significant penalties")
print("   3. Configuration parameters are fully customizable")
print("   4. MCP services enable real-time data enrichment")
print("   5. Historical tracking enables trend analysis")

print("\n🚀 Next Steps:")
print("   • Integrate with live MCP services for real-time data")
print("   • Customize YAML configuration for specific use cases")
print("   • Implement automated rebalancing recommendations")
print("   • Add correlation matrix for better risk assessment")
print("   • Extend asset type categories as needed")

print("\n" + "="*70)
print("🎉 Portfolio Diversification Engine Demo Complete!")
print("="*70)